# Evaluation Dataset Generation for AI Service Pipeline

**Repository**: PTIT_KLTN/AI_Service  
**Purpose**: Generate 3 deterministic evaluation datasets for the AI Service pipeline:  
- Dataset #1: Dish Query Set (in-KB and out-of-KB)
- Dataset #2: Conflict Unit Set
- Dataset #3: Replacement Constraint Set

**Key Properties**:
- Fully offline (no external API calls)
- Deterministic (fixed seeds)
- Reproducible (identical outputs when re-run)
- Outputs in JSONL format with metadata

## Setup and Imports

In [24]:
import json
import random
import sys
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Optional, Set, Tuple
import subprocess
from collections import Counter, defaultdict
import unicodedata
import re

# Add parent directory to path to import AI Service modules
repo_root = Path.cwd().parent
sys.path.insert(0, str(repo_root))

print(f"Repository root: {repo_root}")
print(f"Python version: {sys.version}")
print(f"Current working directory: {Path.cwd()}")

Repository root: d:\DOAN_BT\THUCTAP\AI_Service
Python version: 3.10.0 (tags/v3.10.0:b494f59, Oct  4 2021, 19:00:18) [MSC v.1929 64 bit (AMD64)]
Current working directory: d:\DOAN_BT\THUCTAP\AI_Service\evaluation


## Configuration and Constants

In [25]:
# Fixed seeds for reproducibility
SEED_DISH_QUERY = 2024
SEED_CONFLICT = 2025
SEED_REPLACEMENT = 2026

# Dataset version
DATASET_VERSION = "v1"

# Output directories
EVAL_DIR = Path("datasets")
DISH_QUERY_DIR = EVAL_DIR / "dish_query_set"
CONFLICT_DIR = EVAL_DIR / "conflict_unit_set"
REPLACEMENT_DIR = EVAL_DIR / "replacement_constraint_set"

# Create directories
for dir_path in [DISH_QUERY_DIR, CONFLICT_DIR, REPLACEMENT_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)
    print(f"Created/verified directory: {dir_path}")

# Get git commit hash if available
def get_git_commit():
    try:
        result = subprocess.run(
            ["git", "rev-parse", "HEAD"],
            cwd=repo_root,
            capture_output=True,
            text=True,
            timeout=5
        )
        if result.returncode == 0:
            return result.stdout.strip()[:8]
    except Exception as e:
        print(f"Warning: Could not get git commit: {e}")
    return "unknown"

REPO_COMMIT = get_git_commit()
CREATED_AT = datetime.utcnow().isoformat() + "Z"

print(f"\nMetadata:")
print(f"  Repo commit: {REPO_COMMIT}")
print(f"  Created at: {CREATED_AT}")
print(f"  Dataset version: {DATASET_VERSION}")

Created/verified directory: datasets\dish_query_set
Created/verified directory: datasets\conflict_unit_set
Created/verified directory: datasets\replacement_constraint_set

Metadata:
  Repo commit: e0a7ebe7
  Created at: 2026-02-06T14:57:58.834861Z
  Dataset version: v1


## Load Ground Truth Data from Repository

In [26]:
# Locate data files
DATA_DIR = repo_root / "app" / "data"
DISH_KB_PATH = DATA_DIR / "knowledge_base" / "dish_knowledge_base.json"
INGREDIENT_KB_PATH = DATA_DIR / "knowledge_base" / "ingredient_knowledge_base.json"
CONFLICT_PATH = DATA_DIR / "conflict" / "ingredient_conflict.json"

print("Data file paths:")
print(f"  Dish KB: {DISH_KB_PATH}")
print(f"  Ingredient KB: {INGREDIENT_KB_PATH}")
print(f"  Conflict rules: {CONFLICT_PATH}")

# Verify files exist
for path in [DISH_KB_PATH, INGREDIENT_KB_PATH, CONFLICT_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Required data file not found: {path}")

print("\nAll data files verified!")

Data file paths:
  Dish KB: d:\DOAN_BT\THUCTAP\AI_Service\app\data\knowledge_base\dish_knowledge_base.json
  Ingredient KB: d:\DOAN_BT\THUCTAP\AI_Service\app\data\knowledge_base\ingredient_knowledge_base.json
  Conflict rules: d:\DOAN_BT\THUCTAP\AI_Service\app\data\conflict\ingredient_conflict.json

All data files verified!


In [27]:
# Load Dish Knowledge Base
print("Loading Dish Knowledge Base...")
with open(DISH_KB_PATH, 'r', encoding='utf-8') as f:
    dish_kb = json.load(f)

print(f"Loaded {len(dish_kb)} dishes from KB")
print(f"Sample dish: {dish_kb[0]['name_vi']} (id: {dish_kb[0]['id']})")
print(f"Sample has {len(dish_kb[0]['ingredients'])} ingredients")

Loading Dish Knowledge Base...
Loaded 10741 dishes from KB
Sample dish: Canh nghêu thì là (id: dish0001)
Sample has 11 ingredients


In [28]:
# Load Ingredient Knowledge Base
print("Loading Ingredient Knowledge Base...")
with open(INGREDIENT_KB_PATH, 'r', encoding='utf-8') as f:
    ingredient_kb = json.load(f)

# Build quick lookup by ID and name
ingredient_by_id = {ingr['id']: ingr for ingr in ingredient_kb}
ingredient_by_name_vi = {ingr['name_vi'].lower(): ingr for ingr in ingredient_kb}

# Build category index
category_to_ingredients = defaultdict(list)
for ingr in ingredient_kb:
    if 'category' in ingr and ingr['category']:
        category_to_ingredients[ingr['category']].append(ingr['id'])

print(f"Loaded {len(ingredient_kb)} ingredients from KB")
print(f"Sample ingredient: {ingredient_kb[0]['name_vi']} (id: {ingredient_kb[0]['id']})")
print(f"Total categories: {len(category_to_ingredients)}")
print(f"Sample categories: {list(category_to_ingredients.keys())[:5]}")

Loading Ingredient Knowledge Base...
Loaded 8112 ingredients from KB
Sample ingredient: all purpose cream (id: ingre00001)
Total categories: 20
Sample categories: ['others', 'fruit_jam', 'fresh_meat', 'seafood_&_fish_balls', 'cold_cuts:_sausages_&_ham']


In [29]:
# Load Conflict Rules
print("Loading Conflict Rules...")
with open(CONFLICT_PATH, 'r', encoding='utf-8') as f:
    conflict_rules = json.load(f)

print(f"Loaded {len(conflict_rules)} conflict rules")
print(f"Sample conflict: {conflict_rules[0]['ingre']} <-> {conflict_rules[0]['conflicts']}")
print(f"Severity: {conflict_rules[0]['severity']}")

Loading Conflict Rules...
Loaded 81 conflict rules
Sample conflict: ['Gan heo'] <-> ['Giá đỗ', 'Rau giàu vitamin C']
Severity: medium


## Helper Functions

In [30]:
def normalize_vietnamese_text(text: str) -> str:
    """Normalize Vietnamese text by removing diacritics."""
    # Normalize unicode
    text = unicodedata.normalize('NFD', text)
    # Remove combining characters (diacritics)
    text = ''.join(char for char in text if unicodedata.category(char) != 'Mn')
    # Handle special Vietnamese characters
    text = text.replace('đ', 'd').replace('Đ', 'D')
    return text.lower().strip()

def resolve_ingredient_by_name(name: str) -> Optional[Dict]:
    """Resolve ingredient by Vietnamese name (case-insensitive)."""
    name_lower = name.lower().strip()
    
    # Direct match
    if name_lower in ingredient_by_name_vi:
        return ingredient_by_name_vi[name_lower]
    
    # Try normalized match
    name_norm = normalize_vietnamese_text(name)
    for ingr in ingredient_kb:
        if normalize_vietnamese_text(ingr['name_vi']) == name_norm:
            return ingr
        # Check synonyms
        if 'synonyms' in ingr:
            for syn in ingr['synonyms']:
                if normalize_vietnamese_text(syn) == name_norm:
                    return ingr
    
    return None

def write_jsonl(filepath: Path, records: List[Dict]):
    """Write records to JSONL file (one JSON object per line)."""
    with open(filepath, 'w', encoding='utf-8') as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + '\n')
    print(f"Wrote {len(records)} records to {filepath}")

def write_stats(filepath: Path, stats: Dict):
    """Write statistics to JSON file."""
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(stats, f, ensure_ascii=False, indent=2)
    print(f"Wrote stats to {filepath}")

print("Helper functions defined.")

Helper functions defined.


## Dataset #1: Dish Query Set

Generate two splits:
- **IN-KB**: Dish names that exist in Dish KB (production-style)
- **OUT-OF-KB**: Paraphrased/noisy dish queries (robustness-style)

In [31]:
# Set seed for dish query dataset
random.seed(SEED_DISH_QUERY)

print(f"Generating Dish Query Dataset (seed={SEED_DISH_QUERY})...")
print(f"Total dishes available: {len(dish_kb)}")

Generating Dish Query Dataset (seed=2024)...
Total dishes available: 10741


In [32]:
def get_core_ingredient_ids(dish: Dict) -> List[str]:
    """Extract core ingredient IDs (importance >= 2) from dish."""
    core_ids = []
    for ingr in dish.get('ingredients', []):
        if 'importance' in ingr and ingr['importance'] >= 2:
            core_ids.append(ingr['ingredient_id'])
    return core_ids

def paraphrase_dish_name(dish_name: str, seed_offset: int) -> str:
    """Generate paraphrased version of dish name with Vietnamese variations."""
    rng = random.Random(SEED_DISH_QUERY + seed_offset)
    
    # Vietnamese prefixes/suffixes for natural variation
    prefixes = ["", "Món", "Làm món", "Nấu", "Tôi muốn nấu", "Làm", "Chế biến"]
    suffixes = ["", "ngon", "đơn giản", "tại nhà", "gia đình", "truyền thống"]
    
    # Apply random variation
    variation_type = rng.choice(['prefix', 'suffix', 'typo', 'reorder', 'case'])
    
    if variation_type == 'prefix':
        prefix = rng.choice(prefixes)
        result = f"{prefix} {dish_name}" if prefix else dish_name
    elif variation_type == 'suffix':
        suffix = rng.choice(suffixes)
        result = f"{dish_name} {suffix}" if suffix else dish_name
    elif variation_type == 'typo':
        # Simple character swap for typo
        if len(dish_name) > 3:
            idx = rng.randint(1, len(dish_name) - 2)
            result = dish_name[:idx] + dish_name[idx+1] + dish_name[idx] + dish_name[idx+2:]
        else:
            result = dish_name
    elif variation_type == 'reorder':
        # Reorder words if multi-word
        words = dish_name.split()
        if len(words) >= 2:
            rng.shuffle(words)
            result = ' '.join(words)
        else:
            result = dish_name
    else:  # case
        # Random capitalization
        result = dish_name.title() if rng.random() > 0.5 else dish_name.lower()
    
    return result.strip()

print("Dish query helper functions defined.")

Dish query helper functions defined.


In [33]:
# Generate IN-KB queries
def generate_in_kb_queries(target_base=60, target_excluded=30, target_extra=30, target_both=20):
    """Generate IN-KB dish queries with base, excluded, extra, and both variations."""
    in_kb_queries = []
    case_counter = 1
    
    # Filter dishes with enough ingredients for variations
    valid_dishes = [d for d in dish_kb if len(d.get('ingredients', [])) >= 3]
    
    # Calculate total needed
    total_needed = target_base + target_excluded + target_extra + target_both
    
    # Shuffle to get diverse selection (allow reuse if needed)
    if len(valid_dishes) >= total_needed:
        sampled_dishes = random.sample(valid_dishes, total_needed)
    else:
        # If not enough unique dishes, sample with replacement
        sampled_dishes = random.choices(valid_dishes, k=total_needed)
    
    # Track usage to avoid excessive duplication
    dish_usage_count = Counter()
    
    # Generate base cases
    for dish in sampled_dishes[:target_base]:
        dish_usage_count[dish['id']] += 1
        
        gt_ingredient_ids = [ing['ingredient_id'] for ing in dish['ingredients']]
        core_ids = get_core_ingredient_ids(dish)
        
        query = {
            "case_id": f"DQ_IN_{case_counter:04d}",
            "split": "in_kb",
            "user_input": f"Tôi muốn nấu {dish['name_vi']}",
            "expected": {
                "dish_id": dish['id'],
                "dish_name_vi": dish['name_vi'],
                "gt_ingredient_ids": gt_ingredient_ids,
                "gt_core_ingredient_ids": core_ids,
                "excluded": {"names": [], "ingredient_ids": []},
                "extra": {"names": [], "ingredient_ids": []}
            },
            "tags": ["base"],
            "meta": {
                "seed": SEED_DISH_QUERY,
                "dataset_version": DATASET_VERSION,
                "created_at": CREATED_AT,
                "repo_commit": REPO_COMMIT
            }
        }
        in_kb_queries.append(query)
        case_counter += 1
    
    # Generate excluded cases
    for dish in sampled_dishes[target_base:target_base + target_excluded]:
        dish_usage_count[dish['id']] += 1
        
        gt_ingredient_ids = [ing['ingredient_id'] for ing in dish['ingredients']]
        core_ids = get_core_ingredient_ids(dish)
        
        # Pick a random ingredient from the dish to exclude
        excluded_ing = random.choice(dish['ingredients'])
        excluded_name = excluded_ing['name_vi']
        excluded_id = excluded_ing['ingredient_id']
        
        query = {
            "case_id": f"DQ_IN_{case_counter:04d}",
            "split": "in_kb",
            "user_input": f"Làm {dish['name_vi']}, bỏ {excluded_name}",
            "expected": {
                "dish_id": dish['id'],
                "dish_name_vi": dish['name_vi'],
                "gt_ingredient_ids": gt_ingredient_ids,
                "gt_core_ingredient_ids": core_ids,
                "excluded": {
                    "names": [excluded_name],
                    "ingredient_ids": [excluded_id]
                },
                "extra": {"names": [], "ingredient_ids": []}
            },
            "tags": ["excluded"],
            "meta": {
                "seed": SEED_DISH_QUERY,
                "dataset_version": DATASET_VERSION,
                "created_at": CREATED_AT,
                "repo_commit": REPO_COMMIT
            }
        }
        in_kb_queries.append(query)
        case_counter += 1
    
    # Generate extra cases
    for dish in sampled_dishes[target_base + target_excluded:target_base + target_excluded + target_extra]:
        dish_usage_count[dish['id']] += 1
        
        gt_ingredient_ids = [ing['ingredient_id'] for ing in dish['ingredients']]
        core_ids = get_core_ingredient_ids(dish)
        
        # Pick a random ingredient NOT in the dish
        gt_ids_set = set(gt_ingredient_ids)
        available_extras = [ing for ing in ingredient_kb if ing['id'] not in gt_ids_set]
        
        if available_extras:
            extra_ing = random.choice(available_extras)
            extra_name = extra_ing['name_vi']
            extra_id = extra_ing['id']
            
            query = {
                "case_id": f"DQ_IN_{case_counter:04d}",
                "split": "in_kb",
                "user_input": f"Nấu {dish['name_vi']}, thêm {extra_name}",
                "expected": {
                    "dish_id": dish['id'],
                    "dish_name_vi": dish['name_vi'],
                    "gt_ingredient_ids": gt_ingredient_ids,
                    "gt_core_ingredient_ids": core_ids,
                    "excluded": {"names": [], "ingredient_ids": []},
                    "extra": {
                        "names": [extra_name],
                        "ingredient_ids": [extra_id]
                    }
                },
                "tags": ["extra"],
                "meta": {
                    "seed": SEED_DISH_QUERY,
                    "dataset_version": DATASET_VERSION,
                    "created_at": CREATED_AT,
                    "repo_commit": REPO_COMMIT
                }
            }
            in_kb_queries.append(query)
            case_counter += 1
    
    # Generate both (excluded + extra) cases
    idx = target_base + target_excluded + target_extra
    for dish in sampled_dishes[idx:idx + target_both]:
        if len(dish['ingredients']) < 3:
            continue
            
        dish_usage_count[dish['id']] += 1
        
        gt_ingredient_ids = [ing['ingredient_id'] for ing in dish['ingredients']]
        core_ids = get_core_ingredient_ids(dish)
        
        # Pick excluded from dish
        excluded_ing = random.choice(dish['ingredients'])
        excluded_name = excluded_ing['name_vi']
        excluded_id = excluded_ing['ingredient_id']
        
        # Pick extra NOT in dish
        gt_ids_set = set(gt_ingredient_ids)
        available_extras = [ing for ing in ingredient_kb if ing['id'] not in gt_ids_set]
        
        if available_extras:
            extra_ing = random.choice(available_extras)
            extra_name = extra_ing['name_vi']
            extra_id = extra_ing['id']
            
            query = {
                "case_id": f"DQ_IN_{case_counter:04d}",
                "split": "in_kb",
                "user_input": f"Làm {dish['name_vi']}, bỏ {excluded_name}, thêm {extra_name}",
                "expected": {
                    "dish_id": dish['id'],
                    "dish_name_vi": dish['name_vi'],
                    "gt_ingredient_ids": gt_ingredient_ids,
                    "gt_core_ingredient_ids": core_ids,
                    "excluded": {
                        "names": [excluded_name],
                        "ingredient_ids": [excluded_id]
                    },
                    "extra": {
                        "names": [extra_name],
                        "ingredient_ids": [extra_id]
                    }
                },
                "tags": ["excluded+extra"],
                "meta": {
                    "seed": SEED_DISH_QUERY,
                    "dataset_version": DATASET_VERSION,
                    "created_at": CREATED_AT,
                    "repo_commit": REPO_COMMIT
                }
            }
            in_kb_queries.append(query)
            case_counter += 1
    
    return in_kb_queries, dish_usage_count

# Target: ~750 IN-KB cases (25% of 3000 total)
in_kb_queries, dish_usage = generate_in_kb_queries(
    target_base=300,
    target_excluded=200,
    target_extra=200,
    target_both=50
)
print(f"Generated {len(in_kb_queries)} IN-KB queries")
print(f"Tag distribution: {Counter([q['tags'][0] for q in in_kb_queries])}")

Generated 750 IN-KB queries
Tag distribution: Counter({'base': 300, 'excluded': 200, 'extra': 200, 'excluded+extra': 50})


In [34]:
# Generate OUT-OF-KB queries (paraphrased from IN-KB)
def generate_out_kb_queries(in_kb_queries_sample, target=80):
    """Generate OUT-OF-KB queries by paraphrasing IN-KB queries."""
    out_kb_queries = []
    case_counter = 1
    
    # Take a subset of IN-KB queries to paraphrase
    base_queries = [q for q in in_kb_queries_sample if 'base' in q['tags']]
    
    # If we need more than available base queries, cycle through them
    num_needed = target
    selected_queries = []
    while len(selected_queries) < num_needed:
        remaining = num_needed - len(selected_queries)
        selected_queries.extend(base_queries[:remaining])
    
    for idx, base_query in enumerate(selected_queries):
        # Get dish info
        dish_id = base_query['expected']['dish_id']
        dish_name = base_query['expected']['dish_name_vi']
        
        # Generate paraphrased name
        paraphrased_name = paraphrase_dish_name(dish_name, idx)
        
        # Ensure it's different from original
        if paraphrased_name.lower().strip() == dish_name.lower().strip():
            paraphrased_name = f"Món {dish_name}"
        
        query = {
            "case_id": f"DQ_OUT_{case_counter:04d}",
            "split": "out_kb",
            "user_input": paraphrased_name,
            "expected": base_query['expected'].copy(),
            "tags": ["paraphrased"],
            "meta": {
                "seed": SEED_DISH_QUERY,
                "dataset_version": DATASET_VERSION,
                "created_at": CREATED_AT,
                "repo_commit": REPO_COMMIT,
                "original_dish_name": dish_name
            }
        }
        out_kb_queries.append(query)
        case_counter += 1
    
    return out_kb_queries

# Target: ~750 OUT-OF-KB cases (25% of 3000 total)
out_kb_queries = generate_out_kb_queries(in_kb_queries, target=750)
print(f"Generated {len(out_kb_queries)} OUT-OF-KB queries")

Generated 750 OUT-OF-KB queries


In [35]:
# Calculate statistics for Dish Query Dataset
def calculate_dish_query_stats(queries, split_name):
    """Calculate statistics for dish query dataset."""
    stats = {
        "split": split_name,
        "total_cases": len(queries),
        "tag_distribution": dict(Counter([q['tags'][0] for q in queries])),
    }
    
    # Resolution rates
    excluded_cases = [q for q in queries if q['expected']['excluded']['ingredient_ids']]
    excluded_resolved = sum(1 for q in excluded_cases if q['expected']['excluded']['ingredient_ids'][0])
    stats['excluded_resolution_rate'] = excluded_resolved / len(excluded_cases) if excluded_cases else 0
    
    extra_cases = [q for q in queries if q['expected']['extra']['ingredient_ids']]
    extra_resolved = sum(1 for q in extra_cases if q['expected']['extra']['ingredient_ids'][0])
    stats['extra_resolution_rate'] = extra_resolved / len(extra_cases) if extra_cases else 0
    
    # Ingredient counts
    gt_counts = [len(q['expected']['gt_ingredient_ids']) for q in queries]
    core_counts = [len(q['expected']['gt_core_ingredient_ids']) for q in queries]
    
    stats['avg_gt_ingredient_count'] = sum(gt_counts) / len(gt_counts) if gt_counts else 0
    stats['avg_core_ingredient_count'] = sum(core_counts) / len(core_counts) if core_counts else 0
    
    return stats

in_kb_stats = calculate_dish_query_stats(in_kb_queries, "in_kb")
out_kb_stats = calculate_dish_query_stats(out_kb_queries, "out_kb")

print("\nIN-KB Statistics:")
print(json.dumps(in_kb_stats, indent=2, ensure_ascii=False))
print("\nOUT-OF-KB Statistics:")
print(json.dumps(out_kb_stats, indent=2, ensure_ascii=False))


IN-KB Statistics:
{
  "split": "in_kb",
  "total_cases": 750,
  "tag_distribution": {
    "base": 300,
    "excluded": 200,
    "extra": 200,
    "excluded+extra": 50
  },
  "excluded_resolution_rate": 1.0,
  "extra_resolution_rate": 1.0,
  "avg_gt_ingredient_count": 9.978666666666667,
  "avg_core_ingredient_count": 6.538666666666667
}

OUT-OF-KB Statistics:
{
  "split": "out_kb",
  "total_cases": 750,
  "tag_distribution": {
    "paraphrased": 750
  },
  "excluded_resolution_rate": 0,
  "extra_resolution_rate": 0,
  "avg_gt_ingredient_count": 10.02,
  "avg_core_ingredient_count": 6.569333333333334
}


In [36]:
# Write Dish Query Dataset files
write_jsonl(DISH_QUERY_DIR / "dish_queries_in_kb.jsonl", in_kb_queries)
write_jsonl(DISH_QUERY_DIR / "dish_queries_out_kb.jsonl", out_kb_queries)

combined_stats = {
    "dataset_name": "Dish Query Set",
    "dataset_version": DATASET_VERSION,
    "created_at": CREATED_AT,
    "repo_commit": REPO_COMMIT,
    "seed": SEED_DISH_QUERY,
    "source_files": {
        "dish_kb": str(DISH_KB_PATH),
        "ingredient_kb": str(INGREDIENT_KB_PATH)
    },
    "in_kb": in_kb_stats,
    "out_kb": out_kb_stats
}

write_stats(DISH_QUERY_DIR / "stats.json", combined_stats)
print("\n✅ Dish Query Dataset completed!")

Wrote 750 records to datasets\dish_query_set\dish_queries_in_kb.jsonl
Wrote 750 records to datasets\dish_query_set\dish_queries_out_kb.jsonl
Wrote stats to datasets\dish_query_set\stats.json

✅ Dish Query Dataset completed!


## Dataset #2: Conflict Unit Set

Generate test cases for conflict detection with known conflict pairs.

In [37]:
# Set seed for conflict dataset
random.seed(SEED_CONFLICT)

print(f"Generating Conflict Unit Dataset (seed={SEED_CONFLICT})...")
print(f"Total conflict rules available: {len(conflict_rules)}")

Generating Conflict Unit Dataset (seed=2025)...
Total conflict rules available: 81


In [38]:
# Parse and normalize conflict rules
def parse_conflict_rules():
    """Parse conflict rules and resolve ingredient IDs."""
    parsed_conflicts = []
    
    for rule in conflict_rules:
        severity = rule.get('severity', 'medium')
        reason = rule.get('reason', '')
        
        # Get ingredient A
        ingre_a_names = rule.get('ingre', [])
        conflicts_names = rule.get('conflicts', [])
        
        for a_name in ingre_a_names:
            a_resolved = resolve_ingredient_by_name(a_name)
            
            for b_name in conflicts_names:
                b_resolved = resolve_ingredient_by_name(b_name)
                
                pair = {
                    'a_name': a_name,
                    'a_id': a_resolved['id'] if a_resolved else None,
                    'b_name': b_name,
                    'b_id': b_resolved['id'] if b_resolved else None,
                    'severity': severity,
                    'reason': reason
                }
                parsed_conflicts.append(pair)
    
    return parsed_conflicts

conflict_pairs = parse_conflict_rules()
print(f"Parsed {len(conflict_pairs)} conflict pairs")

# Filter to pairs where both IDs are resolved
resolved_pairs = [p for p in conflict_pairs if p['a_id'] and p['b_id']]
print(f"Resolved pairs (both IDs): {len(resolved_pairs)}")

Parsed 142 conflict pairs
Resolved pairs (both IDs): 73


In [39]:
# Generate conflict test cases
def generate_conflict_cases(target_single=120, target_multi=80):
    """Generate conflict test cases with single and multiple conflict pairs."""
    cases = []
    case_counter = 1
    
    # Determine input format distribution
    format_targets = {
        'name': int(target_single * 0.4),
        'id': int(target_single * 0.4),
        'mixed': int(target_single * 0.2)
    }
    
    # Generate single-pair cases
    for format_type, count in format_targets.items():
        # Sample with replacement if needed
        if count <= len(resolved_pairs):
            sampled_pairs = random.sample(resolved_pairs, count)
        else:
            sampled_pairs = random.choices(resolved_pairs, k=count)
        
        for pair in sampled_pairs:
            if format_type == 'name':
                items = [
                    {"name_vi": pair['a_name'], "ingredient_id": None},
                    {"name_vi": pair['b_name'], "ingredient_id": None}
                ]
            elif format_type == 'id':
                items = [
                    {"name_vi": None, "ingredient_id": pair['a_id']},
                    {"name_vi": None, "ingredient_id": pair['b_id']}
                ]
            else:  # mixed
                items = [
                    {"name_vi": pair['a_name'], "ingredient_id": pair['a_id']},
                    {"name_vi": None, "ingredient_id": pair['b_id']}
                ]
            
            case = {
                "case_id": f"CF_{case_counter:04d}",
                "input_ingredients": {
                    "format": format_type,
                    "items": items
                },
                "expected": {
                    "conflict_pairs": [{
                        "a_id": pair['a_id'],
                        "b_id": pair['b_id'],
                        "severity": pair['severity'],
                        "reason": pair['reason']
                    }],
                    "conflict_count": 1
                },
                "tags": ["single_pair", f"severity_{pair['severity']}"],
                "meta": {
                    "seed": SEED_CONFLICT,
                    "dataset_version": DATASET_VERSION,
                    "created_at": CREATED_AT,
                    "repo_commit": REPO_COMMIT
                }
            }
            cases.append(case)
            case_counter += 1
    
    # Generate multi-pair cases
    for _ in range(target_multi):
        # Sample 2-3 non-overlapping pairs
        num_pairs = random.choice([2, 3])
        selected_pairs = []
        used_ids = set()
        
        attempts = 0
        while len(selected_pairs) < num_pairs and attempts < 100:
            pair = random.choice(resolved_pairs)
            if pair['a_id'] not in used_ids and pair['b_id'] not in used_ids:
                selected_pairs.append(pair)
                used_ids.add(pair['a_id'])
                used_ids.add(pair['b_id'])
            attempts += 1
        
        if len(selected_pairs) >= 2:
            # Build items list
            items = []
            format_type = random.choice(['name', 'id', 'mixed'])
            
            for pair in selected_pairs:
                if format_type == 'name':
                    items.extend([
                        {"name_vi": pair['a_name'], "ingredient_id": None},
                        {"name_vi": pair['b_name'], "ingredient_id": None}
                    ])
                elif format_type == 'id':
                    items.extend([
                        {"name_vi": None, "ingredient_id": pair['a_id']},
                        {"name_vi": None, "ingredient_id": pair['b_id']}
                    ])
                else:  # mixed
                    items.extend([
                        {"name_vi": pair['a_name'], "ingredient_id": pair['a_id']},
                        {"name_vi": None, "ingredient_id": pair['b_id']}
                    ])
            
            expected_pairs = [{
                "a_id": p['a_id'],
                "b_id": p['b_id'],
                "severity": p['severity'],
                "reason": p['reason']
            } for p in selected_pairs]
            
            case = {
                "case_id": f"CF_{case_counter:04d}",
                "input_ingredients": {
                    "format": format_type,
                    "items": items
                },
                "expected": {
                    "conflict_pairs": expected_pairs,
                    "conflict_count": len(selected_pairs)
                },
                "tags": ["multi_pair"],
                "meta": {
                    "seed": SEED_CONFLICT,
                    "dataset_version": DATASET_VERSION,
                    "created_at": CREATED_AT,
                    "repo_commit": REPO_COMMIT
                }
            }
            cases.append(case)
            case_counter += 1
    
    return cases

# Target: ~800 cases (27% of 3000 total)
conflict_cases = generate_conflict_cases(target_single=480, target_multi=320)
print(f"Generated {len(conflict_cases)} conflict test cases")

Generated 800 conflict test cases


In [40]:
# Calculate statistics for Conflict Dataset
def calculate_conflict_stats(cases):
    """Calculate statistics for conflict dataset."""
    stats = {
        "total_cases": len(cases),
        "tag_distribution": {},
        "format_distribution": {},
        "severity_distribution": {},
        "avg_conflicts_per_case": 0,
        "resolution_rate": 0
    }
    
    # Tag distribution
    tags = []
    for case in cases:
        tags.extend(case['tags'])
    tag_counter = Counter(tags)
    stats['tag_distribution'] = dict(tag_counter)
    
    # Format distribution
    formats = [case['input_ingredients']['format'] for case in cases]
    stats['format_distribution'] = dict(Counter(formats))
    
    # Severity distribution
    severities = []
    for case in cases:
        for pair in case['expected']['conflict_pairs']:
            severities.append(pair['severity'])
    stats['severity_distribution'] = dict(Counter(severities))
    
    # Average conflicts
    conflict_counts = [case['expected']['conflict_count'] for case in cases]
    stats['avg_conflicts_per_case'] = sum(conflict_counts) / len(conflict_counts) if conflict_counts else 0
    
    # Resolution rate (both IDs resolved)
    resolved_count = 0
    total_pairs = 0
    for case in cases:
        for pair in case['expected']['conflict_pairs']:
            total_pairs += 1
            if pair['a_id'] and pair['b_id']:
                resolved_count += 1
    stats['resolution_rate'] = resolved_count / total_pairs if total_pairs else 0
    
    return stats

conflict_stats = calculate_conflict_stats(conflict_cases)
print("\nConflict Dataset Statistics:")
print(json.dumps(conflict_stats, indent=2, ensure_ascii=False))


Conflict Dataset Statistics:
{
  "total_cases": 800,
  "tag_distribution": {
    "single_pair": 480,
    "severity_low": 450,
    "severity_medium": 30,
    "multi_pair": 320
  },
  "format_distribution": {
    "name": 303,
    "id": 300,
    "mixed": 197
  },
  "severity_distribution": {
    "low": 1214,
    "medium": 81
  },
  "avg_conflicts_per_case": 1.61875,
  "resolution_rate": 1.0
}


In [41]:
# Write Conflict Dataset files
write_jsonl(CONFLICT_DIR / "conflict_unit_tests.jsonl", conflict_cases)

conflict_full_stats = {
    "dataset_name": "Conflict Unit Set",
    "dataset_version": DATASET_VERSION,
    "created_at": CREATED_AT,
    "repo_commit": REPO_COMMIT,
    "seed": SEED_CONFLICT,
    "source_files": {
        "conflict_rules": str(CONFLICT_PATH),
        "ingredient_kb": str(INGREDIENT_KB_PATH)
    },
    **conflict_stats
}

write_stats(CONFLICT_DIR / "stats.json", conflict_full_stats)
print("\n✅ Conflict Unit Dataset completed!")

Wrote 800 records to datasets\conflict_unit_set\conflict_unit_tests.jsonl
Wrote stats to datasets\conflict_unit_set\stats.json

✅ Conflict Unit Dataset completed!


## Dataset #3: Replacement Constraint Set

Generate test cases for replacement suggestion constraints.

In [42]:
# Set seed for replacement dataset
random.seed(SEED_REPLACEMENT)

print(f"Generating Replacement Constraint Dataset (seed={SEED_REPLACEMENT})...")

Generating Replacement Constraint Dataset (seed=2026)...


In [43]:
# Generate replacement constraint cases
def generate_replacement_cases(target=150):
    """Generate replacement constraint test cases."""
    cases = []
    case_counter = 1
    
    # Use resolved conflict pairs
    available_pairs = [p for p in conflict_pairs if p['a_id'] and p['b_id']]
    
    for _ in range(target):
        # Select a conflict pair (with replacement if needed)
        pair = random.choice(available_pairs)
        
        # Randomly choose which ingredient to replace
        if random.random() > 0.5:
            target_id = pair['a_id']
            conflict_id = pair['b_id']
        else:
            target_id = pair['b_id']
            conflict_id = pair['a_id']
        
        # Get target ingredient info
        target_ingr = ingredient_by_id.get(target_id)
        if not target_ingr:
            continue
        
        target_category = target_ingr.get('category')
        
        # Build exclude list (conflict pair + 0-2 random ingredients)
        exclude_ids = [target_id, conflict_id]
        num_extra_excludes = random.randint(0, 2)
        
        for _ in range(num_extra_excludes):
            random_ingr = random.choice(ingredient_kb)
            if random_ingr['id'] not in exclude_ids:
                exclude_ids.append(random_ingr['id'])
        
        # Check if valid replacements exist in ontology
        valid_replacement_exists = False
        if target_category and target_category in category_to_ingredients:
            category_ingredients = category_to_ingredients[target_category]
            available = [ing_id for ing_id in category_ingredients if ing_id not in exclude_ids]
            valid_replacement_exists = len(available) >= 1
        
        case = {
            "case_id": f"RP_{case_counter:04d}",
            "context": {
                "dish_id": None,
                "dish_name_vi": None,
                "conflicted_pair": {
                    "a_id": pair['a_id'],
                    "b_id": pair['b_id']
                },
                "target_replace_id": target_id,
                "target_category": target_category,
                "exclude_ids": exclude_ids
            },
            "constraints": {
                "same_category": True,
                "must_not_include_ids": exclude_ids,
                "unique": True,
                "max_suggestions": 3
            },
            "expected": {
                "valid_replacement_exists_in_ontology": valid_replacement_exists,
                "min_valid_suggestions": 1 if valid_replacement_exists else 0
            },
            "tags": ["from_conflict_rules", "category_based"],
            "meta": {
                "seed": SEED_REPLACEMENT,
                "dataset_version": DATASET_VERSION,
                "created_at": CREATED_AT,
                "repo_commit": REPO_COMMIT
            }
        }
        cases.append(case)
        case_counter += 1
    
    return cases

# Target: ~700 cases (23% of 3000 total)
replacement_cases = generate_replacement_cases(target=700)
print(f"Generated {len(replacement_cases)} replacement constraint cases")

Generated 700 replacement constraint cases


In [44]:
# Calculate statistics for Replacement Dataset
def calculate_replacement_stats(cases):
    """Calculate statistics for replacement dataset."""
    stats = {
        "total_cases": len(cases),
        "category_resolved_rate": 0,
        "valid_replacement_exists_rate": 0,
        "edge_case_rate": 0,
        "max_suggestions_distribution": {}
    }
    
    # Category resolution
    category_resolved = sum(1 for c in cases if c['context']['target_category'])
    stats['category_resolved_rate'] = category_resolved / len(cases) if cases else 0
    
    # Valid replacement exists
    valid_exists = sum(1 for c in cases if c['expected']['valid_replacement_exists_in_ontology'])
    stats['valid_replacement_exists_rate'] = valid_exists / len(cases) if cases else 0
    
    # Edge cases (no valid replacement)
    edge_cases = sum(1 for c in cases if not c['expected']['valid_replacement_exists_in_ontology'])
    stats['edge_case_rate'] = edge_cases / len(cases) if cases else 0
    
    # Max suggestions distribution
    max_suggs = [c['constraints']['max_suggestions'] for c in cases]
    stats['max_suggestions_distribution'] = dict(Counter(max_suggs))
    
    return stats

replacement_stats = calculate_replacement_stats(replacement_cases)
print("\nReplacement Dataset Statistics:")
print(json.dumps(replacement_stats, indent=2, ensure_ascii=False))


Replacement Dataset Statistics:
{
  "total_cases": 700,
  "category_resolved_rate": 1.0,
  "valid_replacement_exists_rate": 1.0,
  "edge_case_rate": 0.0,
  "max_suggestions_distribution": {
    "3": 700
  }
}


In [45]:
# Write Replacement Dataset files
write_jsonl(REPLACEMENT_DIR / "replacement_cases.jsonl", replacement_cases)

replacement_full_stats = {
    "dataset_name": "Replacement Constraint Set",
    "dataset_version": DATASET_VERSION,
    "created_at": CREATED_AT,
    "repo_commit": REPO_COMMIT,
    "seed": SEED_REPLACEMENT,
    "source_files": {
        "conflict_rules": str(CONFLICT_PATH),
        "ingredient_kb": str(INGREDIENT_KB_PATH)
    },
    **replacement_stats
}

write_stats(REPLACEMENT_DIR / "stats.json", replacement_full_stats)
print("\n✅ Replacement Constraint Dataset completed!")

Wrote 700 records to datasets\replacement_constraint_set\replacement_cases.jsonl
Wrote stats to datasets\replacement_constraint_set\stats.json

✅ Replacement Constraint Dataset completed!


## Final Summary

In [46]:
print("="*80)
print("EVALUATION DATASET GENERATION COMPLETED")
print("="*80)
print(f"\nRepository: PTIT_KLTN/AI_Service")
print(f"Dataset version: {DATASET_VERSION}")
print(f"Created at: {CREATED_AT}")
print(f"Git commit: {REPO_COMMIT}")
print(f"\nOutput directory: {EVAL_DIR.absolute()}")

print(f"\n{'Dataset':<30} {'Files':<40} {'Cases':>8}")
print("-" * 80)

# Dataset 1
print(f"{'1. Dish Query Set':<30} {'dish_queries_in_kb.jsonl':<40} {len(in_kb_queries):>8}")
print(f"{'':<30} {'dish_queries_out_kb.jsonl':<40} {len(out_kb_queries):>8}")
print(f"{'':<30} {'stats.json':<40} {'':>8}")

# Dataset 2
print(f"{'2. Conflict Unit Set':<30} {'conflict_unit_tests.jsonl':<40} {len(conflict_cases):>8}")
print(f"{'':<30} {'stats.json':<40} {'':>8}")

# Dataset 3
print(f"{'3. Replacement Constraint':<30} {'replacement_cases.jsonl':<40} {len(replacement_cases):>8}")
print(f"{'':<30} {'stats.json':<40} {'':>8}")

print("-" * 80)
print(f"{'TOTAL':<30} {'':<40} {len(in_kb_queries) + len(out_kb_queries) + len(conflict_cases) + len(replacement_cases):>8}")

print(f"\n✅ All datasets generated successfully!")
print(f"\nNote: All outputs are deterministic and reproducible with fixed seeds.")
print(f"Re-running this notebook will produce identical results.")

EVALUATION DATASET GENERATION COMPLETED

Repository: PTIT_KLTN/AI_Service
Dataset version: v1
Created at: 2026-02-06T14:57:58.834861Z
Git commit: e0a7ebe7

Output directory: d:\DOAN_BT\THUCTAP\AI_Service\evaluation\datasets

Dataset                        Files                                       Cases
--------------------------------------------------------------------------------
1. Dish Query Set              dish_queries_in_kb.jsonl                      750
                               dish_queries_out_kb.jsonl                     750
                               stats.json                                       
2. Conflict Unit Set           conflict_unit_tests.jsonl                     800
                               stats.json                                       
3. Replacement Constraint      replacement_cases.jsonl                       700
                               stats.json                                       
----------------------------------------------